# 03 — Model Eğitimi (184 Sınıf)
226 sınıftan validation accuracy < %50 olan 42 sınıf çıkarıldı.
Kalan 184 sınıfla yeniden eğitim yapılıyor.

**Beklenen sonuç:** Top-1 ~%75-80, Top-3 ~%90+

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import top_k_accuracy_score
from sklearn.preprocessing import LabelEncoder

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
mixed_precision.set_global_policy('mixed_float16')

In [ ]:
# ── CONFIG ────────────────────────────────────────────────
BASE       = '/content/drive/MyDrive/sign-language-project'
PACKED_DIR = f'{BASE}/packed_landmarks'
SIGN_CSV   = f'{BASE}/dataset/SignList_ClassId_TR_EN.csv'
SAVE_PATH  = f'{BASE}/best_model_184.keras'
DEMO_DIR   = f'{BASE}/demo_assets_184'

SEQ_LEN    = 16
FEAT_DIM   = 252
NUM_CLASSES = 184
BATCH_SIZE = 32
EPOCHS     = 100
PATIENCE   = 15

# Val accuracy >= 50% olan sınıflar (226 model per_class_accuracy.csv'den alındı)
# Bu listeyi per_class_accuracy.csv'den güncelle
SELECTED_CLASSES_CSV = f'{BASE}/per_class_accuracy.csv'
THRESHOLD = 0.50

print('Config OK')

In [ ]:
# ── VERİ YÜKLE ────────────────────────────────────────────
X_train = np.load(f'{PACKED_DIR}/train_X.npy')
y_train = np.load(f'{PACKED_DIR}/train_y.npy')
X_val   = np.load(f'{PACKED_DIR}/val_X.npy')
y_val   = np.load(f'{PACKED_DIR}/val_y.npy')

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_val:   {X_val.shape},   y_val:   {y_val.shape}')
assert not np.isnan(X_train).any(), 'NaN var!'
print('✅ Veri temiz')

In [ ]:
# ── SINIF FİLTRELEME ──────────────────────────────────────
per_class_df = pd.read_csv(SELECTED_CLASSES_CSV)
good_classes = per_class_df[per_class_df['accuracy'] >= THRESHOLD]
selected_classes = sorted(good_classes['class_id'].tolist())

print(f'Threshold {THRESHOLD*100:.0f}%: {len(selected_classes)} sınıf seçildi')

# Sıfır sample'ları filtrele
nonzero_mask     = ~np.all(X_train == 0, axis=(1,2))
nonzero_mask_val = ~np.all(X_val   == 0, axis=(1,2))
X_train_nz = X_train[nonzero_mask]
y_train_nz = y_train[nonzero_mask]
X_val_nz   = X_val[nonzero_mask_val]
y_val_nz   = y_val[nonzero_mask_val]

# Seçilen sınıfları filtrele
train_mask = np.isin(y_train_nz, selected_classes)
val_mask   = np.isin(y_val_nz,   selected_classes)
X_train_sel = X_train_nz[train_mask]
y_train_sel = y_train_nz[train_mask]
X_val_sel   = X_val_nz[val_mask]
y_val_sel   = y_val_nz[val_mask]

print(f'Train: {X_train_sel.shape}')
print(f'Val:   {X_val_sel.shape}')

# Label remap (0'dan başlat)
le = LabelEncoder()
le.fit(selected_classes)
y_train_remap = le.transform(y_train_sel)
y_val_remap   = le.transform(y_val_sel)

print(f'Label aralığı: {y_train_remap.min()} - {y_train_remap.max()}')
print(f'Unique sınıf: {len(np.unique(y_train_remap))}')

# LabelEncoder'ı kaydet (demo için lazım)
os.makedirs(DEMO_DIR, exist_ok=True)
np.save(f'{DEMO_DIR}/label_encoder_classes.npy', le.classes_)
print('LabelEncoder kaydedildi')

In [ ]:
# ── NORMALİZASYON ─────────────────────────────────────────
feat_mean = X_train_sel.mean(axis=(0,1))
feat_std  = X_train_sel.std(axis=(0,1))
feat_std  = np.where(feat_std < 1e-6, 1.0, feat_std)

X_train_norm = (X_train_sel - feat_mean) / feat_std
X_val_norm   = (X_val_sel   - feat_mean) / feat_std

print(f'Norm mean range: [{feat_mean.min():.3f}, {feat_mean.max():.3f}]')
print(f'X_train_norm mean: {X_train_norm.mean():.3f}')
assert not np.isnan(X_train_norm).any(), 'NaN var!'

# Kaydet
norm_stats = {'mean': feat_mean.tolist(), 'std': feat_std.tolist()}
with open(f'{DEMO_DIR}/norm_stats.json', 'w') as f:
    json.dump(norm_stats, f)
print('norm_stats.json kaydedildi')

In [ ]:
# ── DATA AUGMENTATION ─────────────────────────────────────
@tf.function
def augment(x, y):
    if tf.random.uniform(()) > 0.5:
        noise = tf.random.normal(tf.shape(x), stddev=0.02)
        x = x + noise

    if tf.random.uniform(()) > 0.7:
        mask_len = tf.random.uniform((), 1, 3, dtype=tf.int32)
        start    = tf.random.uniform((), 0, SEQ_LEN - mask_len, dtype=tf.int32)
        mask     = tf.concat([
            tf.ones([start, FEAT_DIM]),
            tf.zeros([mask_len, FEAT_DIM]),
            tf.ones([SEQ_LEN - start - mask_len, FEAT_DIM])
        ], axis=0)
        x = x * tf.cast(mask, tf.float32)

    if tf.random.uniform(()) > 0.5:
        scale = tf.random.uniform((), 0.9, 1.1)
        x = x * scale

    return x, y

print('Augmentation hazır')

In [ ]:
# ── tf.data PIPELINE ──────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(X, y, training=False, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices(
        (X.astype(np.float32), y.astype(np.int32))
    )
    if training:
        ds = ds.shuffle(buffer_size=len(X), reshuffle_each_iteration=True)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(X_train_norm, y_train_remap, training=True)
val_ds   = make_dataset(X_val_norm,   y_val_remap,   training=False)
print('Dataset hazır')

In [ ]:
# ── MODEL ─────────────────────────────────────────────────
def build_model(seq_len=16, feat_dim=252, num_classes=184):
    inputs = keras.Input(shape=(seq_len, feat_dim))

    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True))(inputs)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=False))(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)

    return keras.Model(inputs, x)

model = build_model()
model.summary()
print(f'Toplam parametre: {model.count_params():,}')

In [ ]:
# ── CLASS WEIGHT ──────────────────────────────────────────
classes = np.arange(NUM_CLASSES)
cw = compute_class_weight('balanced', classes=classes, y=y_train_remap)
cw = np.clip(cw, 0.1, 10.0)
class_weight_dict = {i: cw[i] for i in range(NUM_CLASSES)}
print(f'Class weight — min: {cw.min():.3f}, max: {cw.max():.3f}')

In [ ]:
# ── COMPILE ───────────────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print('Model compile edildi')

In [ ]:
# ── CALLBACKS ─────────────────────────────────────────────
callbacks = [
    keras.callbacks.ModelCheckpoint(
        SAVE_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    keras.callbacks.CSVLogger(f'{BASE}/training_log_184.csv'),
]
print(f'Model kaydedilecek: {SAVE_PATH}')

In [ ]:
# ── EĞİTİM ────────────────────────────────────────────────
print('Eğitim başlıyor...')
print(f'  Train: {len(X_train_norm)} örnek, {NUM_CLASSES} sınıf')
print(f'  Val:   {len(X_val_norm)} örnek')
print(f'  Batch: {BATCH_SIZE}, Max epoch: {EPOCHS}')

start = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

print(f'Eğitim tamamlandı: {(time.time()-start)/60:.1f} dakika')

with open(f'{BASE}/training_history_184.json', 'w') as f:
    json.dump({
        'accuracy':     history.history.get('accuracy', []),
        'val_accuracy': history.history.get('val_accuracy', []),
        'loss':         history.history.get('loss', []),
        'val_loss':     history.history.get('val_loss', [])
    }, f)

In [ ]:
# ── DEĞERLENDİRME ─────────────────────────────────────────
best_model = keras.models.load_model(SAVE_PATH)
y_pred_proba = best_model.predict(X_val_norm, verbose=1)

print('\n' + '='*50)
print('SONUÇLAR (184 sınıf)')
print('='*50)

top1 = top_k_accuracy_score(y_val_remap, y_pred_proba, k=1)
top3 = top_k_accuracy_score(y_val_remap, y_pred_proba, k=3)
top5 = top_k_accuracy_score(y_val_remap, y_pred_proba, k=5)

print(f'Top-1 Accuracy: {top1:.4f} ({top1*100:.2f}%)')
print(f'Top-3 Accuracy: {top3:.4f} ({top3*100:.2f}%)')
print(f'Top-5 Accuracy: {top5:.4f} ({top5*100:.2f}%)')

In [ ]:
# ── CLASS BAZLI ANALİZ ────────────────────────────────────
sign_df   = pd.read_csv(SIGN_CSV)
label_map = dict(zip(sign_df['ClassId'], sign_df['TR']))

y_pred = np.argmax(y_pred_proba, axis=1)

per_class = []
for new_label in range(NUM_CLASSES):
    orig_label = le.inverse_transform([new_label])[0]
    mask = y_val_remap == new_label
    if mask.sum() == 0:
        continue
    acc = (y_pred[mask] == new_label).mean()
    per_class.append({
        'class_id': int(orig_label),
        'TR':       label_map.get(orig_label, '?'),
        'n_val':    int(mask.sum()),
        'accuracy': float(acc)
    })

per_class_df = pd.DataFrame(per_class).sort_values('accuracy', ascending=False)
per_class_df.to_csv(f'{BASE}/per_class_accuracy_184.csv', index=False)

print('En iyi 20 sınıf:')
print(per_class_df.head(20).to_string(index=False))
print('\nEn kötü 20 sınıf:')
print(per_class_df.tail(20).to_string(index=False))

In [ ]:
# ── DEMO DOSYALARINI KAYDET ───────────────────────────────
import shutil

# Model kopyala
shutil.copy(SAVE_PATH, f'{DEMO_DIR}/model.keras')

# Label map — sadece seçilen 184 sınıf, yeni index ile
label_map_demo = {}
for new_label in range(NUM_CLASSES):
    orig_label = int(le.inverse_transform([new_label])[0])
    row = sign_df[sign_df['ClassId'] == orig_label]
    if len(row) > 0:
        label_map_demo[new_label] = {
            'original_class_id': orig_label,
            'TR': str(row.iloc[0]['TR']),
            'EN': str(row.iloc[0]['EN'])
        }

with open(f'{DEMO_DIR}/label_map.json', 'w', encoding='utf-8') as f:
    json.dump(label_map_demo, f, ensure_ascii=False, indent=2)

# Demo config
demo_config = {
    'seq_len':    SEQ_LEN,
    'feat_dim':   FEAT_DIM,
    'num_classes': NUM_CLASSES,
    'landmark_layout': 'color_left_hand(63)+color_right_hand(63)+depth_left_hand(63)+depth_right_hand(63)',
    'model_file':       'model.keras',
    'norm_stats_file':  'norm_stats.json',
    'label_map_file':   'label_map.json',
    'confidence_threshold': 0.4,
    'top_k_display': 3
}
with open(f'{DEMO_DIR}/demo_config.json', 'w') as f:
    json.dump(demo_config, f, indent=2)

print(f'\n✅ Demo dosyaları kaydedildi: {DEMO_DIR}')
print('  - model.keras')
print('  - norm_stats.json')
print('  - label_map.json')
print('  - demo_config.json')
print('  - label_encoder_classes.npy')
print('\nBu dosyaları bilgisayarına indirip web demo klasörüne koy.')